## Scalable path (`tensor_network_updates`)

The original `pstrings_to_mpo_optimized` builds the *exact* MPO (bond dimension
= number of terms `K`) via dense `(2,2,K,K)` diagonal tensors before truncating,
so it needs `O(K**2)` memory and dies for large `K`.

`tensor_network_updates` instead does a single left-to-right compression sweep
that never forms the full `K`-bond: memory is `O(max_bond * K)` and time is
`O(n_qubits * K * max_bond**2)`. This scales to ~40 qubits and
hundreds-of-thousands to millions of terms.

In [ ]:
from symmer.operators import PauliwordOp
from symmer.approximate.tensor_network_updates import (
    mpo_from_pauliwordop,
    find_groundstate_quimb,
)
import numpy as np

# large qubit count + many terms
Hq = PauliwordOp.random(n_qubits=40, n_terms=200_000, complex_coeffs=False, density=0.6)

## add a dominant term to make ground state easy to find... so looking at
## bitstrings is more interesting
Hq += PauliwordOp.from_list([Hq.n_qubits * "Z"], [10_000])


# PauliwordOp -> compressed MPO (bounded bond dimension, complex64 to save memory)
MPO = mpo_from_pauliwordop(Hq, max_bond=100, cutoff=1e-10, dtype=np.complex64)
MPO

In [ ]:
# solve for the ground-state energy with DMRG.
# For 40 qubits do NOT densify the state (2**40 amplitudes); keep the MPS.
result = find_groundstate_quimb(
    MPO,
    bond_dims=[20, 50, 100, 100],
    cutoffs=1e-8,
    max_sweeps=8,
    verbosity=1,
)
print("ground-state energy:", result["energy"])

In [ ]:
result["mps"]

### Reading off the highest-amplitude bitstrings

The ground state is an MPS, so the full `2**n` vector is never formed. To find
the most-probable computational-basis bitstrings we search the MPS directly:

* `method='beam'` (default) — beam search over the most-probable prefixes.
  Always fast (`O(n * beam_width * D**2)`) and terminating; exact for peaked
  states (increase `beam_width` for more accuracy).
* `method='exact'` — best-first branch-and-bound with the admissible bound
  `sqrt(P(prefix))`; provably returns the true top-`k`, but is only efficient
  for peaked / low-entanglement states.

`bitstring[i]` is the value of qubit `i` (same ordering as the Pauli strings).

In [ ]:
from symmer.approximate.tensor_network_updates import top_amplitude_bitstrings

# top-k most probable bitstrings straight from the MPS (no dense state)
top = top_amplitude_bitstrings(result["mps"], k=10, beam_width=512)

print(f"{'probability':>14}   {'amplitude':>22}   bitstring")
for bitstring, amp, prob in top:
    print(f"{prob:14.3e}   {amp:22.5f}   {bitstring}")